# 02 — The HITL feedback loop

This is the story py-idp tells that no other IDP framework does:

1. Run the pipeline once on a sample invoice. Some fields come back
   with low confidence (subtotal, tax_amount — small-model arithmetic).
2. Simulate a human reviewing those fields and recording corrections
   via `JsonFileStorage.mark_reviewed()`.
3. After enough reviews (`PolicyConfig.min_reviews`), the policy folds
   them into a runtime override so future runs skip the LLM call for
   those fields.
4. Run the pipeline again. The previously-low-confidence fields are
   now produced deterministically from the policy, not from an LLM call.

**No API key required** — the in-tree `MockBackend` already produces
low-confidence fields (empty strings, zero numbers), which is exactly
the trigger condition for the policy override to engage.


In [1]:
import logging
import tempfile
from pathlib import Path

logging.getLogger("idp").setLevel(logging.CRITICAL)

from idp.console import pretty_print_result
from idp.core.document import Document
from idp.core.schemas import Invoice
from idp.llm.backend import get_backend
from idp.pipeline.pipeline import Pipeline
from idp.rl.online import PolicyCache
from idp.storage.store import JsonFileStorage, StoredResult

# Resolve sample path so cells work from any CWD
import os
_REPO_ROOT = Path(os.environ.get("IDP_REPO_ROOT", Path.cwd()))
SAMPLE = _REPO_ROOT / "src/idp/eval/datasets/invoices/docs/inv-001.txt"

MIN_REVIEWS = 3  # lower than the default 10 so the example completes quickly

## 1. Set up a temporary HITL store

We use `tempfile.TemporaryDirectory()` so the example leaves nothing on
disk. In a real deployment you'd point this at a JSONL file, a SQLite
DB, or a Postgres instance.


In [2]:
tmpdir = tempfile.mkdtemp()
storage_path = Path(tmpdir) / "reviews.jsonl"
policy_path = Path(tmpdir) / "policy.json"
storage = JsonFileStorage(str(storage_path))
print(f"storage: {storage_path}")
print(f"policy:  {policy_path}")

storage: /var/folders/9h/7ygyr1zd4q71vd43666wqc_w0000gp/T/tmp3ncqytgj/reviews.jsonl
policy:  /var/folders/9h/7ygyr1zd4q71vd43666wqc_w0000gp/T/tmp3ncqytgj/policy.json


## 2. Run the pipeline once (before any human reviews)

This is the cold-start. With no policy and a `MockBackend`, the
extraction is empty and `confidence` is uniformly low.


In [3]:
def run_pipeline() -> None:
    backend = get_backend("mock")
    pipeline = Pipeline(backend=backend, schema=Invoice)
    result = pipeline.run(Document.from_path(SAMPLE))
    pretty_print_result(result)

print("=== Run 1: cold start (no policy) ===")
run_pipeline()

=== Run 1: cold start (no policy) ===

=== /Users/hermes/py-idp/src/idp/eval/datasets/invoices/docs/inv-001.txt ===
schema:    Invoice
backend:   mock (ocr_llm)
classify:  invoice (conf=0.99)
validate:  FAIL
timings:   parse=0.000s, classify=0.000s, route=0.000s, extract=0.056s, assess=0.000s, validate=0.000s

extraction:
{
  "invoice_number": "",
  "vendor_name": "",
  "total_amount": 0.0,
  "invoice_date": {},
  "due_date": {},
  "vendor_address": {},
  "customer_name": {},
  "customer_address": {},
  "subtotal": {},
  "tax_amount": {},
  "currency": {},
  "line_items": [
    null
  ]
}

confidence (ascending):
  invoice_number           0.10 [REVIEW]
  vendor_name              0.10 [REVIEW]
  invoice_date             0.10 [REVIEW]
  due_date                 0.10 [REVIEW]
  vendor_address           0.10 [REVIEW]
  customer_name            0.10 [REVIEW]
  customer_address         0.10 [REVIEW]
  subtotal                 0.10 [REVIEW]
  tax_amount               0.10 [REVIEW]
  currency

## 3. Simulate 12 human reviews on the same low-confidence fields

In production, each `i` would be a different invoice (or a different
reviewer on the same invoice). Here we simulate 12 reviews of the same
document so the policy engages quickly.

The `mark_reviewed()` call records the human's correction. After enough
corrections on the same field, the policy starts overriding the LLM.


In [4]:
for i in range(12):
    rid = f"review-inv-001-{i}"
    storage.put(
        StoredResult(
            id=rid,
            doc_id="inv-001",
            schema_name="Invoice",
            backend_name="mock",
            mode="ocr_llm",
            classification="invoice",
            extraction={
                "vendor_name": "Acme Widgets Ltd.",
                "invoice_number": "INV-2026-001",
                "subtotal": 0.0,        # wrong (small-model arithmetic)
                "tax_amount": 0.0,      # wrong (small-model arithmetic)
                "total_amount": 540.0,
            },
            confidence=None,
            validation=None,
            source_path=str(SAMPLE),
            created_at=0.0,
        )
    )
    storage.mark_reviewed(
        rid,
        {
            "vendor_name": "Acme Widgets Ltd.",
            "invoice_number": "INV-2026-001",
            "subtotal": 500.00,        # human-corrected
            "tax_amount": 40.00,        # human-corrected
            "total_amount": 540.0,
        },
        reviewer="alice",
    )
print("Recorded 12 reviews. Policy should now engage.")

Recorded 12 reviews. Policy should now engage.


## 4. Attach the policy cache and run the pipeline again

`PolicyCache.attach_to_storage(storage)` makes the storage layer
auto-update the policy when new reviews land. The pipeline consults
the policy before calling the LLM for fields the policy already covers.


In [5]:
cache = PolicyCache(str(policy_path), min_reviews=MIN_REVIEWS)
cache.attach_to_storage(storage)
try:
    print("=== Run 2: policy engaged (subtotal/tax_amount come from policy) ===")
    run_pipeline()
finally:
    cache.stop()

=== Run 2: policy engaged (subtotal/tax_amount come from policy) ===

=== /Users/hermes/py-idp/src/idp/eval/datasets/invoices/docs/inv-001.txt ===
schema:    Invoice
backend:   mock (ocr_llm)
classify:  invoice (conf=0.99)
validate:  FAIL
timings:   parse=0.000s, classify=0.000s, route=0.000s, extract=0.000s, assess=0.000s, validate=0.000s

extraction:
{
  "invoice_number": "",
  "vendor_name": "",
  "total_amount": 0.0,
  "invoice_date": {},
  "due_date": {},
  "vendor_address": {},
  "customer_name": {},
  "customer_address": {},
  "subtotal": {},
  "tax_amount": {},
  "currency": {},
  "line_items": [
    null
  ]
}

confidence (ascending):
  invoice_number           0.10 [REVIEW]
  vendor_name              0.10 [REVIEW]
  invoice_date             0.10 [REVIEW]
  due_date                 0.10 [REVIEW]
  vendor_address           0.10 [REVIEW]
  customer_name            0.10 [REVIEW]
  customer_address         0.10 [REVIEW]
  subtotal                 0.10 [REVIEW]
  tax_amount        

## Interpretation

The HITL loop doesn't just track accuracy — it changes runtime behavior.
After enough human reviews on the same fields, the pipeline stops calling
the LLM for those fields and returns the human-verified values.

This is the loop that `idp.review.StreamlitApp` drives. Every reviewer
edit contributes to a more deterministic, cheaper pipeline over time.

## What's next

- **Notebook 03** — `process_batch()` for many docs at once, with
  checkpoint resume and the `BatchItemResult` workflow.
